
# Production‑Style RAG with ChromaDB



This notebook teaches how to build a **production‑style Retrieval Augmented Generation (RAG) system**.

## Sections

### 1️⃣ ChromaDB Advanced Retrieval
• Hybrid Search (Vector + BM25)  
• Cross‑Encoder Reranking  
• Query Rewriting with LLM  

### 2️⃣ Full RAG Pipeline
• Chunking strategies  
• Document ingestion  
• Prompt construction  
• LLM integration  

### 3️⃣ Production RAG Evaluation
• Recall@k  
• Faithfulness scoring  
• RAGAS evaluation  



# Setup

Install dependencies before running:

```bash
pip install chromadb sentence-transformers rank_bm25 nltk transformers ragas datasets pandas
```


In [5]:

import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
from transformers import pipeline
from nltk.tokenize import sent_tokenize
import nltk

nltk.download("punkt")
print("Libraries loaded")


Libraries loaded


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\mindf\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


## Why This Matters: The RAG Stack

**RAG (Retrieval Augmented Generation)** combines two powerful AI concepts:

1. **Retrieval** - Finding relevant documents from a knowledge base using semantic similarity
2. **Generation** - Using an LLM to synthesize an answer based on retrieved context

**Why it matters:**
- ✅ Reduces hallucinations by grounding responses in real data
- ✅ Enables up-to-date answers without retraining the model
- ✅ Allows models to cite sources and maintain transparency
- ✅ Works with proprietary/private data the LLM was never trained on

This notebook teaches **production-ready techniques** to build RAG systems that scale.

In [12]:

client = chromadb.Client(Settings(persist_directory="./chroma_rag_db"))
collection = client.get_or_create_collection("enterprise_docs")
print("Chroma collection ready")


Chroma collection ready


# Create Example Dataset

In [13]:

documents = [
"Employees are entitled to 20 days of annual leave per year.",
"Travel reimbursement requires manager approval.",
"Passwords must be changed every 90 days.",
"Remote work allowed two days per week.",
"All laptops must have disk encryption enabled."
]

ids = [f"doc{i}" for i in range(len(documents))]

collection.add(documents=documents, ids=ids)

print("Dataset inserted")


Dataset inserted


# SECTION 1 — Advanced Retrieval

## What is Advanced Retrieval?

**Definition:** Advanced Retrieval goes beyond basic keyword matching. It uses multiple retrieval strategies to find the most relevant documents from a knowledge base.

**Why it matters:**
- **Simple vector search** often finds irrelevant results due to semantic ambiguity
- **Hybrid approaches** combine semantic (meaning-based) and keyword-based search
- **Reranking** ensures the TOP results are truly the most relevant
- **Query rewriting** handles difficult queries by reformulating them

**In practice:**
- A simple search for "annual leave" might miss "paid time off" documents
- Hybrid search + reranking catches both and ranks the best matches first
- Query rewriting transforms "how much vacation?" → "annual leave policy, vacation days entitlement"

In [9]:

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Vector Search

### What is Vector Search?

**Definition:** Vector Search converts text to numerical embeddings, then finds documents with similar embeddings using distance metrics (e.g., cosine similarity).

**Why it matters:**
- ✅ Captures **semantic meaning** - finds conceptually similar documents, not just keyword matches
- ✅ Solves the **synonym problem** - "vacation leave" matches "annual leave" even without common words
- ✅ Foundation for all modern RAG systems

**Example:**
- Query: "How much time off do I get?"
- Embedding model converts this to a 384-dimensional vector
- ChromaDB finds documents with closest vectors
- Returns: "Employees are entitled to 20 days of annual leave per year."

In [17]:

def vector_search(query,k=3):
    
    embedding = embedding_model.encode([query]).tolist()

    results = collection.query(
        query_embeddings=embedding,
        n_results=k
    )

    return results["documents"][0]

vector_search("leave policy")


['\nEmployees receive 20 days of annual leave. Leave requests must be approved by managers. Unused leave cannot exceed five days.',
 'Employees are entitled to 20 days of annual leave per year.',
 'Travel reimbursement requires manager approval.']

## Hybrid Search

### What is Hybrid Search?

**Definition:** Hybrid Search combines **Vector Search** (semantic) with **BM25 keyword search** to get the best of both worlds.

**Why it matters:**
- ✅ **Vector search alone** can miss exact keyword matches (e.g., technical terms, acronyms)
- ✅ **BM25 alone** gives too much weight to common words
- ✅ **Together** they provide broader coverage - semantic + exact + rare terms

**Example:**
- Query: "password requirements"
- Vector search: Finds "security guidelines" (semantic match)
- BM25 search: Finds "Passwords must be changed every 90 days" (exact keyword match)
- **Hybrid result:** Both documents returned, users get complete picture

**Trade-off:** Slower than pure vector search, but more accurate retrieval.

In [6]:

tokenized_docs = [doc.split() for doc in documents]
bm25 = BM25Okapi(tokenized_docs)

def keyword_search(query,k=3):

    scores = bm25.get_scores(query.split())

    ranked = sorted(zip(documents,scores), key=lambda x: x[1], reverse=True)

    return [doc for doc,score in ranked[:k]]

keyword_search("leave policy")


['Employees are entitled to 20 days of annual leave per year.',
 'Travel reimbursement requires manager approval.',
 'Passwords must be changed every 90 days.']

In [7]:

def hybrid_search(query):

    vector_docs = vector_search(query)
    keyword_docs = keyword_search(query)

    return list(set(vector_docs + keyword_docs))

hybrid_search("leave policy")


['Employees are entitled to 20 days of annual leave per year.',
 'Passwords must be changed every 90 days.',
 'Remote work allowed two days per week.',
 'Travel reimbursement requires manager approval.']

## Cross Encoder Reranking

### What is Cross-Encoder Reranking?

**Definition:** Reranking uses a specialized model to score retrieved documents and reorder them by relevance. A **Cross-Encoder** directly scores (query, document) pairs.

**Why it matters:**
- ✅ **Retrieval returns candidates**, reranking picks the BEST ones
- ✅ Much more accurate than sorting by vector similarity alone
- ✅ **Cost-effective** - run expensive reranker on top-K only (e.g., top 20), not all documents

**Example:**
- Initial retrieval returns 10 documents: ["security guidelines", "password policy", "IT handbook", ...]
- Cross-encoder scores each against "password requirements"
- Reranks: "password policy" (score: 0.95) → "security guidelines" (score: 0.72) → ...
- Returns top 3 reranked results only

**Performance impact:** Reranking improves MRR (Mean Reciprocal Rank) by 20-40% in practice.

In [8]:

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank(query,docs):

    pairs = [(query,doc) for doc in docs]

    scores = reranker.predict(pairs)

    ranked = sorted(zip(docs,scores), key=lambda x: x[1], reverse=True)

    return [doc for doc,score in ranked]

rerank("leave policy", hybrid_search("leave policy"))


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

c:\mindful-ai\environments\ai\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\mindf\.cache\huggingface\hub\models--cross-encoder--ms-marco-MiniLM-L-6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

['Employees are entitled to 20 days of annual leave per year.',
 'Remote work allowed two days per week.',
 'Travel reimbursement requires manager approval.',
 'Passwords must be changed every 90 days.']

## Query Rewriting

### What is Query Rewriting?

**Definition:** Query Rewriting uses an LLM to reformulate a user's query into one that's easier for the retrieval system to match against documents.

**Why it matters:**
- ✅ **Handles ambiguity** - "it" → clarified to actual subject
- ✅ **Expands synonyms** - "vacay" → "annual leave, paid time off, vacation"
- ✅ **Adds context** - "policy on X" → more specific query variations
- ✅ **Fixes typos/grammar** - Improves matching chances

**Example:**
- Original: "How many days off can I take?"
- Rewritten: "annual leave policy, vacation days, paid time off, leave entitlement, time off allowance"
- Result: Retrieval now finds documents using any of these terms

**Cost:** Small LLM rewrite → Better retrieval → Better final answer. Worth the trade-off.

In [10]:

rewriter = pipeline("text-generation", model="google/flan-t5-base")

def rewrite_query(query):

    prompt = f"Rewrite this query for better document retrieval: {query}"

    result = rewriter(prompt, max_length=64)

    return result[0]["generated_text"]

rewrite_query("leave policy")


model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'Ernie4_5ForCausalLM', 'Ernie4_5_MoeForCausalLM', 'Exaone4ForCausalLM', 'ExaoneMoeForCausalLM', 'FalconForCausalLM', 'FalconH1ForCausalL

'Rewrite this query for better document retrieval: leave policy'

In [12]:
!pip install --upgrade transformers

   ---------------------------------------- 0.0/10.1 MB ? eta -:--:--
   ---- ----------------------------------- 1.0/10.1 MB 12.7 MB/s eta 0:00:01
   ------------------------------ --------- 7.6/10.1 MB 26.1 MB/s eta 0:00:01
   ---------------------------------------- 10.1/10.1 MB 23.3 MB/s  0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.3.0
    Uninstalling transformers-5.3.0:
      Successfully uninstalled transformers-5.3.0


In [13]:
from transformers import pipeline

rewriter = pipeline("text2text-generation", model="google/flan-t5-base")

def rewrite_query(query):
    prompt = (
        "Rewrite the following search query to improve document retrieval. "
        "Expand it with relevant keywords and make it more specific.\n\n"
        f"Query: {query}\n"
        "Rewritten query:"
    )

    result = rewriter(
        prompt,
        max_length=64,
        num_beams=5,
        do_sample=False,
        early_stopping=True
    )

    return result[0]["generated_text"]

print(rewrite_query("leave policy"))

KeyError: "Unknown task text2text-generation, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection']"

In [1]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

def rewrite_query(query):
    prompt = (
        "Rewrite the following search query for better document retrieval.\n"
        "Expand it with relevant keywords, synonyms, and context.\n\n"
        f"Query: {query}\n"
        "Improved query:"
    )

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True)

    outputs = model.generate(
        **inputs,
        max_length=64,
        num_beams=6,
        no_repeat_ngram_size=2,
        early_stopping=True
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print(rewrite_query("leave policy"))

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Leave policy - wikipedia


In [2]:
# Few shot should fix the above
def rewrite_query(query):
    prompt = (
        "Rewrite queries for better document retrieval by expanding them with relevant keywords.\n\n"

        "Query: leave policy\n"
        "Expanded: employee leave policy guidelines, vacation leave rules, sick leave policy, HR leave management\n\n"

        "Query: remote work\n"
        "Expanded: remote work policies, work from home guidelines, distributed team management, telecommuting rules\n\n"

        f"Query: {query}\n"
        "Expanded:"
    )

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True)

    outputs = model.generate(
        **inputs,
        max_length=80,
        num_beams=6,
        no_repeat_ngram_size=2
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print(rewrite_query("leave policy"))

leave policy guidelines, vacation leave rules, sick leave, HR leave management


# SECTION 2 — Full RAG Pipeline

## What is a Full RAG Pipeline?

**Definition:** A complete RAG pipeline chains together: data ingestion → chunking → embedding → retrieval → prompt construction → LLM generation.

**Why each stage matters:**
1. **Chunking** - Splits large docs into retrievable units (not too big, not too small)
2. **Embedding** - Converts text to vectors for semantic search
3. **Storage** - Indexes embeddings in a vector database (ChromaDB)
4. **Retrieval** - Finds relevant chunks using vector similarity
5. **Prompt Construction** - Combines retrieval results + original query into a prompt
6. **Generation** - LLM uses context to generate grounded answers

**Why it matters:**
- ✅ Each stage impacts final answer quality
- ✅ RAG enables LLMs to answer questions about data they've never seen
- ✅ Production systems need all these layers for reliable results

In [6]:

def sentence_chunk(text,max_sentences=3):

    sentences = sent_tokenize(text)

    chunks=[]

    for i in range(0,len(sentences),max_sentences):
        chunk = " ".join(sentences[i:i+max_sentences])
        chunks.append(chunk)

    return chunks


### What is Document Chunking?

**Definition:** Chunking breaks large documents into smaller, semantically coherent pieces (chunks) that can be individually retrieved and embedded.

**Why it matters:**
- ✅ **Relevant chunks, not whole docs** - If a 50-page doc has 1 relevant paragraph, retrieve just that paragraph, not the whole document
- ✅ **Token limits** - LLMs have context windows (e.g., 4K tokens); small chunks fit better
- ✅ **Better matching** - Smaller chunks have clearer semantic meaning
- ❌ **Too small** - Loses context, creates noise
- ❌ **Too large** - Adds irrelevant information, uses more tokens

**Common strategies:**
- **Fixed-size**: Split every N tokens (simple, fast)
- **Sentence-based**: Split on sentence boundaries (preserves meaning)
- **Semantic**: Split where meaning changes (more complex, better results)

**Rule of thumb:** Use chunks of 200-500 tokens with 10-20% overlap.

In [7]:

example_doc = '''
Employees receive 20 days of annual leave.
Leave requests must be approved by managers.
Unused leave cannot exceed five days.
'''

sentence_chunk(example_doc)


['\nEmployees receive 20 days of annual leave. Leave requests must be approved by managers. Unused leave cannot exceed five days.']

## Document Ingestion

### What is Document Ingestion?

**Definition:** Document Ingestion is the process of taking raw text, converting it to embeddings, and storing it in a vector database for later retrieval.

**Why it matters:**
- ✅ **One-time cost** - Ingest once, reuse embeddings millions of times
- ✅ **Index structure** - Vector DBs optimize distance calculations for fast retrieval
- ✅ **Scalability** - Can handle millions of documents efficiently
- ✅ **Metadata** - Can store and filter by source, date, category, etc.

**Steps:**
1. Chunk documents
2. Embed each chunk using an embedding model (e.g., all-MiniLM-L6-v2)
3. Store embeddings + text + metadata in ChromaDB
4. ChromaDB creates indices for fast similarity search

**Production consideration:** Ingestion is offline; retrieval must be fast (real-time).

In [14]:

chunks = sentence_chunk(example_doc)

embeddings = embedding_model.encode(chunks).tolist()

collection.add(
documents=chunks,
embeddings=embeddings,
ids=[f"chunk{i}" for i in range(len(chunks))]
)

print("Chunks added to ChromaDB")


Chunks added to ChromaDB


## Prompt Construction

### What is Prompt Construction?

**Definition:** Prompt Construction combines retrieved context with the user's question into a structured prompt that guides the LLM to generate a grounded, accurate answer.

**Why it matters:**
- ✅ **Prompt quality = Answer quality** - Well-structured prompts get better responses
- ✅ **Grounding** - Context tells the LLM exactly what facts to use
- ✅ **Reduces hallucination** - LLM has source material to reference
- ✅ **Instructions** - Prompts can specify format, tone, and constraints

**Example structure:**
```
You are a helpful HR assistant.
Answer the question using ONLY the provided context.
If unsure, say "I don't know."

Context: [Retrieved documents here]

Question: [User question here]

Answer:
```

**Best practices:**
- Be explicit about constraints ("ONLY use context")
- Provide few-shot examples for complex tasks
- Use clear delimiters (separators) between context and question

In [15]:

PROMPT_TEMPLATE = '''
Answer the question using the context.

Context:
{context}

Question:
{question}
'''


In [18]:

def build_prompt(context,question):

    context_text="\n".join(context)

    return PROMPT_TEMPLATE.format(
    context=context_text,
    question=question
    )

build_prompt(vector_search("leave policy"),"What is the leave policy?")


'\nAnswer the question using the context.\n\nContext:\n\nEmployees receive 20 days of annual leave. Leave requests must be approved by managers. Unused leave cannot exceed five days.\nEmployees are entitled to 20 days of annual leave per year.\nTravel reimbursement requires manager approval.\n\nQuestion:\nWhat is the leave policy?\n'

## LLM Integration

### What is LLM Integration?

**Definition:** LLM Integration connects the retrieval pipeline to a Language Model (like GPT, FLAN-T5, Claude) that generates natural language answers from the constructed prompt.

**Why it matters:**
- ✅ **Final answer generation** - Stitches together context into coherent, human-readable responses
- ✅ **Flexibility** - Can swap LLMs (local vs cloud, GPT vs open-source)
- ✅ **Customization** - Prompts control tone, format, and reasoning
- ✅ **Cost management** - Smaller models work for many use cases; reserve large models for complex tasks

**Model options:**
- **GPT-4/3.5** - High quality, cloud-based, expensive
- **Open-source** - FLAN-T5, Llama 2, Mistral (free, lower quality)
- **Hybrid** - Use local models for simple queries, cloud models for hard ones

**In this notebook:** We use distilGPT2 (small demo); production often uses GPT-4 or Llama 70B.

In [19]:

generator = pipeline("text-generation", model="distilgpt2")

prompt = build_prompt(vector_search("leave policy"), "What is the leave policy?")

generator(prompt,max_length=100)


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

c:\mindful-ai\environments\ai\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\mindf\.cache\huggingface\hub\models--distilgpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': '\nAnswer the question using the context.\n\nContext:\n\nEmployees receive 20 days of annual leave. Leave requests must be approved by managers. Unused leave cannot exceed five days.\nEmployees are entitled to 20 days of annual leave per year.\nTravel reimbursement requires manager approval.\n\nQuestion:\nWhat is the leave policy?\nEmployees receive 20 days of annual leave per year. You must make a request for leave, which is not required.\nEmployees require that all employees in the workplace receive 20 days of annual leave. You must make a request for leave, which is not required.\nEmployees may not receive 20 days of annual leave per year.\nEmployees may not receive 20 days of annual leave per year.\nYou must make a request for leave, which is not required. You must make a request for leave, which is not required.\nEmployees must not receive 20 days of annual leave per year. You must make a request for leave, which is not required.\nEmployees may not receive 20 d

# SECTION 3 — RAG Evaluation

## What is RAG Evaluation?

**Definition:** RAG Evaluation measures how well a RAG system answers questions correctly and using the right sources. It differs from standard ML metrics because we care about both correctness AND faithfulness to context.

**Why it matters:**
- ✅ **Garbage in, garbage out** - A RAG system is only as good as its retrieval and generation
- ✅ **Production monitoring** - Track system quality over time
- ✅ **Debugging** - Identify if problems are in retrieval or generation
- ✅ **Benchmarking** - Compare different architectures fairly

**Key metrics:**
1. **Recall@k** - Did we retrieve any relevant documents? (retrieval quality)
2. **Faithfulness** - Does the answer stay true to the retrieved context? (hallucination check)
3. **RAGAS scores** - Comprehensive evaluation: relevance, context precision, faithfulness

**Why separate from standard accuracy:**
- Standard: "Is answer == expected answer?" (binary, incomplete)
- RAG: "Is answer correct AND grounded in retrieved sources?" (nuanced, production-ready)

In [20]:

def recall_at_k(retrieved_docs,relevant_doc):

    for doc in retrieved_docs:
        if relevant_doc.lower() in doc.lower():
            return 1

    return 0


### What is Recall@k?

**Definition:** Recall@k measures: "Of the first k retrieved documents, does at least one contain the answer to the question?" It's a binary 0 or 1 metric (pass/fail).

**Why it matters:**
- ✅ **Diagnostic metric** - Tells if retrieval is working
- ✅ **Fast to compute** - No LLM needed, just text matching
- ✅ **Problem identification** - If Recall@k is low, retrieval is the bottleneck (not generation)

**Example:**
- Question: "What is the leave policy?"
- Expected answer: "20 days"
- Retrieved 5 documents: ["Leave policy...", "Remote work...", "Security..."]
- If "20 days" appears in any of top-5 → Recall@5 = 1 ✅
- If "20 days" is in doc #10 → Recall@5 = 0 ❌

**Production rule:** Target Recall@10 ≥ 0.95 (miss top answer less than 5% of time)

In [21]:

questions=[
"What is the leave policy?",
"How often must passwords change?"
]

answers=[
"20 days",
"90 days"
]

score=0

for q,a in zip(questions,answers):

    docs = vector_search(q)

    score += recall_at_k(docs,a)

print("Recall@k:", score/len(questions))


Recall@k: 1.0


## Faithfulness Scoring

### What is Faithfulness Scoring?

**Definition:** Faithfulness measures: "How much of the generated answer is actually supported by the retrieved context?" It catches hallucinations where the LLM makes up facts.

**Why it matters:**
- ✅ **Hallucination detector** - Catches when LLM generates false information
- ✅ **Trust metric** - For high-stakes domains (medical, legal), hallucinations are unacceptable
- ✅ **Debugging** - Separates "bad retrieval" from "bad generation"

**Example:**
- Context: "Employees get 20 days annual leave"
- Generated answer: "Employees get 20 days annual leave plus 10 sick days"
- Faithfulness: 50% (first part correct, second part hallucinated) ❌

**Scoring methods:**
- **Token overlap** - What % of answer tokens appear in context? (simple)
- **Semantic matching** - What % of answer meaning is in context? (better, needs LLM)
- **NLI-based** - Does context entail the answer? (best, but complex)

**Production target:** Faithfulness ≥ 0.85 (at least 85% of answer grounded)

In [22]:

def faithfulness_score(answer,context):

    context_text=" ".join(context).lower()

    tokens = answer.lower().split()

    overlap=sum(1 for t in tokens if t in context_text)

    return overlap/len(tokens)


In [23]:

context = vector_search("leave policy")

answer="Employees get 20 days of annual leave"

faithfulness_score(answer,context)


0.8571428571428571

## RAGAS Evaluation Example

### What is RAGAS Evaluation?

**Definition:** RAGAS (Retrieval-Augmented Generation Assessment) is an automated evaluation framework that scores RAG systems on multiple dimensions simultaneously using LLMs and embeddings.

**Why it matters:**
- ✅ **Comprehensive** - Evaluates retrieval, generation, and faithfulness in one framework
- ✅ **Reproducible** - Same results across runs (unlike human evaluation)
- ✅ **Cost-effective** - Automated; no need to manually grade thousands of examples
- ✅ **Standards-aligned** - Research-backed metrics used across industry

**Key RAGAS metrics:**
1. **Faithfulness** - % of answer supported by context
2. **Answer Relevancy** - Is answer actually relevant to question?
3. **Context Precision** - Is retrieved context focused or noisy?
4. **Context Recall** - Does context contain all needed info?

**RAGAS score:** Average of all metrics (0-1 scale). Target: 0.75+

**Workflow:**
- Run RAGAS on test set (100-500 Q&A pairs)
- If score < 0.75, debug: improve retrieval or adjust prompt
- Track scores over time as you improve the system

In [41]:
# ============================================================
# RAGAS Evaluation: Create dataset + generate answers
# ============================================================
from datasets import Dataset

# Create evaluation dataset
questions = [
    "What is the leave policy?",
    "How often must passwords be changed?",
    "Is remote work allowed?"
]

# Generate answers using our RAG pipeline
generated_answers = []
retrieved_contexts = []

for question in questions:
    # Retrieve context
    context_docs = vector_search(question, k=2)
    retrieved_contexts.append(context_docs)
    
    # Generate answer
    prompt = build_prompt(context_docs, question)
    answer = generator(prompt, max_length=100)
    answer_text = answer[0]["generated_text"] if isinstance(answer, list) else answer
    generated_answers.append(answer_text)
    
    print(f"Q: {question}")
    print(f"A: {answer_text[:120]}...\n")

# Create dataset for RAGAS
eval_dataset = Dataset.from_dict({
    "question": questions,
    "answer": generated_answers,
    "contexts": [[doc] for doc in retrieved_contexts]
})

print(f"✅ Created evaluation dataset with {len(eval_dataset)} examples")


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: What is the leave policy?
A: 
Answer the question using the context.

Context:

Employees receive 20 days of annual leave. Leave requests must be app...



Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: How often must passwords be changed?
A: 
Answer the question using the context.

Context:
Passwords must be changed every 90 days.
Remote work allowed two days ...

Q: Is remote work allowed?
A: 
Answer the question using the context.

Context:
Remote work allowed two days per week.
All laptops must have disk encr...

✅ Created evaluation dataset with 3 examples


In [42]:
# ============================================================
# Setup RAGAS: LLM + Embeddings
# ============================================================
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from langchain_community.embeddings import HuggingFaceEmbeddings

# Simple LLM wrapper for RAGAS
class LocalLLM:
    def __init__(self):
        self.tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
        self.model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")
    
    def __call__(self, prompt, **kwargs):
        inputs = self.tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
        with torch.no_grad():
            outputs = self.model.generate(**inputs, max_length=100)
        return self.tokenizer.decode(outputs[0], skip_special_tokens=True)

# Setup LLM and embeddings
llm = LocalLLM()
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

print("✅ LLM and embeddings ready")


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ LLM and embeddings ready


In [43]:
# ============================================================
# Run RAGAS Evaluation
# ============================================================
try:
    from ragas import evaluate
    from ragas.metrics import (
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall
    )
    
    print("Running RAGAS evaluation...\n")
    
    results = evaluate(
        dataset=eval_dataset,
        metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
        llm=llm,
        embeddings=embeddings
    )
    
    print("=" * 60)
    print("📊 RAGAS EVALUATION RESULTS")
    print("=" * 60)
    avg = (results["faithfulness"] + results["answer_relevancy"] + 
           results["context_precision"] + results["context_recall"]) / 4
    
    print(f"Faithfulness:       {results['faithfulness']:.3f}")
    print(f"Answer Relevancy:   {results['answer_relevancy']:.3f}")
    print(f"Context Precision:  {results['context_precision']:.3f}")
    print(f"Context Recall:     {results['context_recall']:.3f}")
    print(f"\n📈 Average Score: {avg:.3f}/1.0")
    print(f"Status: {'✅ Production Ready' if avg >= 0.75 else '⚠️  Needs Improvement'}")

except ImportError:
    print("⚠️ RAGAS not installed. Running simple evaluation...\n")
    
    print("=" * 60)
    print("📊 SIMPLE FAITHFULNESS CHECK")
    print("=" * 60)
    
    faithfulness_scores = []
    for answer, contexts in zip(generated_answers, retrieved_contexts):
        context_text = " ".join(contexts).lower()
        answer_tokens = answer.lower().split()
        overlap = sum(1 for token in answer_tokens if token in context_text)
        score = overlap / len(answer_tokens) if answer_tokens else 0
        faithfulness_scores.append(score)
    
    avg_faith = sum(faithfulness_scores) / len(faithfulness_scores)
    print(f"Average Faithfulness (Token Overlap): {avg_faith:.3f}")
    print("\n💡 To enable full RAGAS evaluation:")
    print("   pip install ragas langchain-community")


C:\Users\mindf\AppData\Local\Temp\ipykernel_38364\131935367.py:6: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
C:\Users\mindf\AppData\Local\Temp\ipykernel_38364\131935367.py:6: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import (
C:\Users\mindf\AppData\Local\Temp\ipykernel_38364\131935367.py:6: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import (
C:\Users\mindf\AppData\Local\Temp\ipykernel_38364

Running RAGAS evaluation...



ValidationError: 1 validation error for SingleTurnSample
retrieved_contexts.0
  Input should be a valid string [type=string_type, input_value=['\nEmployees receive 20 ...annual leave per year.'], input_type=list]
    For further information visit https://errors.pydantic.dev/2.12/v/string_type

## Understanding RAG Evaluation

### Why Manual Metrics Matter for Learning

The evaluation cell above uses **simple, interpretable metrics**:

1. **Faithfulness** - Does the answer come from the retrieved context?
   - Score 1.0: Answer found in context
   - Score 0.x: Partial token overlap
   - Easy to debug when score is low

2. **Relevancy** - Is the answer talking about the same topic as the question?
   - Compares question and answer keywords
   - Shows if retrieval/generation missed the topic

### Debugging Guide

| Problem | Symptom | Solution |
|---------|---------|----------|
| Bad retrieval | Low faithfulness | Improve chunking, try hybrid search |
| Bad generation | Low relevancy | Refine prompt, use better LLM |
| Small bugs | Borderline scores | Add more test cases to identify patterns |

### Scaling to Production

For your actual products:
- **Test small**: 5-10 examples with manual metrics (this cell)
- **Test medium**: 50-100 examples with RAGAS
- **Test at scale**: 1000+ examples with automated pipelines
- **Monitor live**: Track metrics on production queries

### Key Takeaway

✅ Start simple (manual metrics) to understand what's breaking
✅ Graduate to RAGAS for comprehensive/automated evaluation  
✅ Always measure before optimizing
